In [1]:
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [2]:
import streamlit

print(
    "Streamlit version:",
    streamlit.__version__
)

Streamlit version: 1.51.0


In [3]:
%%writefile fraud_dashboard.py

import pandas as pd
import streamlit as st

from pathlib import Path

st.set_page_config(
    page_title="Fraud Detection Dashboard",
    page_icon="🛡️",
    layout="wide"
)

BASE_FOLDER = Path(__file__).resolve().parent

TRANSACTION_FILE = (
    BASE_FOLDER /
    "kafka_transaction_results.csv"
)

FRAUD_ALERT_FILE = (
    BASE_FOLDER /
    "kafka_fraud_alerts.csv"
)

MONITORING_FILE = (
    BASE_FOLDER /
    "kafka_monitoring_alerts.csv"
)

ALERT_HISTORY_FILE = (
    BASE_FOLDER /
    "fraud_alert_history.csv"
)

METRICS_FILE = (
    BASE_FOLDER /
    "real_time_stream_metrics.csv"
)

def read_csv_safely(file_path):

    if not file_path.exists():
        return pd.DataFrame()

    try:

        return pd.read_csv(file_path)

    except pd.errors.EmptyDataError:

        return pd.DataFrame()

    except Exception as error:

        st.warning(
            f"Could not read {file_path.name}: "
            f"{error}"
        )

        return pd.DataFrame()


def prepare_transaction_data(data):

    if data.empty:
        return data

    data = data.copy()

    numeric_columns = [
        "fraud_probability",
        "risk_score",
        "predicted_fraud"
    ]

    for column in numeric_columns:

        if column in data.columns:

            data[column] = pd.to_numeric(
                data[column],
                errors="coerce"
            )

    date_columns = [
        "produced_at",
        "processed_at"
    ]

    for column in date_columns:

        if column in data.columns:

            data[column] = pd.to_datetime(
                data[column],
                errors="coerce"
            )

    return data

transaction_data = prepare_transaction_data(
    read_csv_safely(
        TRANSACTION_FILE
    )
)

fraud_alerts = read_csv_safely(
    FRAUD_ALERT_FILE
)

monitoring_alerts = read_csv_safely(
    MONITORING_FILE
)

alert_history = read_csv_safely(
    ALERT_HISTORY_FILE
)

model_metrics = read_csv_safely(
    METRICS_FILE
)

st.sidebar.title(
    "🛡️ Fraud Detection"
)

selected_page = st.sidebar.selectbox(
    "Select dashboard page",
    [
        "Overview",
        "Transactions",
        "Fraud Alerts",
        "Alert History",
        "Model Performance"
    ]
)

if st.sidebar.button(
    "Refresh Dashboard",
    use_container_width=True
):

    st.rerun()

st.sidebar.markdown("---")

st.sidebar.caption(
    "Kafka fraud monitoring dashboard"
)

if selected_page == "Overview":

    st.title(
        "Real-Time Fraud Detection Dashboard"
    )

    st.write(
        "Monitor transaction risk, fraud alerts "
        "and investigation activity."
    )

    if transaction_data.empty:

        st.warning(
            "No Kafka transaction results found. "
            "Run the Kafka producer and consumer first."
        )

    else:

        total_transactions = len(
            transaction_data
        )

        if "predicted_fraud" in transaction_data.columns:

            total_fraud_alerts = int(
                transaction_data[
                    "predicted_fraud"
                ]
                .fillna(0)
                .sum()
            )

        else:

            total_fraud_alerts = 0

        if "risk_level" in transaction_data.columns:

            monitoring_count = int(
                transaction_data[
                    "risk_level"
                ]
                .isin(
                    [
                        "Medium",
                        "High",
                        "Critical"
                    ]
                )
                .sum()
            )

            critical_count = int(
                (
                    transaction_data[
                        "risk_level"
                    ] == "Critical"
                ).sum()
            )

        else:

            monitoring_count = 0
            critical_count = 0

        column1, column2, column3, column4 = (
            st.columns(4)
        )

        column1.metric(
            "Transactions",
            total_transactions
        )

        column2.metric(
            "Confirmed Alerts",
            total_fraud_alerts
        )

        column3.metric(
            "Monitoring Cases",
            monitoring_count
        )

        column4.metric(
            "Critical Cases",
            critical_count
        )

        st.subheader(
            "Risk-Level Distribution"
        )

        if "risk_level" in transaction_data.columns:

            risk_order = [
                "Critical",
                "High",
                "Medium",
                "Low"
            ]

            risk_counts = (
                transaction_data[
                    "risk_level"
                ]
                .value_counts()
                .reindex(
                    risk_order,
                    fill_value=0
                )
            )

            st.bar_chart(
                risk_counts
            )

        st.subheader(
            "Highest-Risk Transactions"
        )

        if "risk_score" in transaction_data.columns:

            highest_risk = (
                transaction_data
                .sort_values(
                    by="risk_score",
                    ascending=False
                )
                .head(10)
            )

            st.dataframe(
                highest_risk,
                use_container_width=True,
                hide_index=True
            )

elif selected_page == "Transactions":

    st.title(
        "Transaction Monitoring"
    )

    if transaction_data.empty:

        st.warning(
            "No transaction data available."
        )

    else:

        filtered_transactions = (
            transaction_data.copy()
        )

        if "risk_level" in transaction_data.columns:

            available_levels = sorted(
                transaction_data[
                    "risk_level"
                ]
                .dropna()
                .astype(str)
                .unique()
                .tolist()
            )

            selected_levels = st.multiselect(
                "Filter by risk level",
                available_levels,
                default=available_levels
            )

            filtered_transactions = (
                filtered_transactions[
                    filtered_transactions[
                        "risk_level"
                    ].isin(selected_levels)
                ]
            )

        if "risk_score" in transaction_data.columns:

            minimum_risk = st.slider(
                "Minimum risk score",
                min_value=0,
                max_value=100,
                value=0
            )

            filtered_transactions = (
                filtered_transactions[
                    filtered_transactions[
                        "risk_score"
                    ] >= minimum_risk
                ]
            )

        st.write(
            "Transactions found:",
            len(filtered_transactions)
        )

        st.dataframe(
            filtered_transactions,
            use_container_width=True,
            hide_index=True
        )

        st.download_button(
            label="Download Filtered Transactions",
            data=filtered_transactions.to_csv(
                index=False
            ),
            file_name=(
                "filtered_fraud_transactions.csv"
            ),
            mime="text/csv"
        )


elif selected_page == "Fraud Alerts":

    st.title(
        "Fraud Alerts and Monitoring"
    )

    alert_tab, monitoring_tab = st.tabs(
        [
            "Confirmed Fraud",
            "Monitoring Cases"
        ]
    )

    with alert_tab:

        st.subheader(
            "Confirmed Fraud Alerts"
        )

        if fraud_alerts.empty:

            st.info(
                "No transaction crossed the "
                "optimized fraud threshold."
            )

        else:

            st.dataframe(
                fraud_alerts,
                use_container_width=True,
                hide_index=True
            )

            st.download_button(
                "Download Fraud Alerts",
                fraud_alerts.to_csv(
                    index=False
                ),
                "fraud_alerts.csv",
                "text/csv"
            )

    with monitoring_tab:

        st.subheader(
            "Transactions Requiring Monitoring"
        )

        if monitoring_alerts.empty:

            st.info(
                "No monitoring cases are available."
            )

        else:

            if "risk_score" in monitoring_alerts.columns:

                monitoring_alerts = (
                    monitoring_alerts
                    .sort_values(
                        by="risk_score",
                        ascending=False
                    )
                )

            st.dataframe(
                monitoring_alerts,
                use_container_width=True,
                hide_index=True
            )

            st.download_button(
                "Download Monitoring Cases",
                monitoring_alerts.to_csv(
                    index=False
                ),
                "monitoring_alerts.csv",
                "text/csv"
            )

elif selected_page == "Alert History":

    st.title(
        "Automated Notification History"
    )

    if alert_history.empty:

        st.info(
            "No automated notification "
            "history is available."
        )

    else:

        history_column1, history_column2 = (
            st.columns(2)
        )

        history_column1.metric(
            "Total Notifications",
            len(alert_history)
        )

        if "stream_id" in alert_history.columns:

            unique_alerts = (
                alert_history[
                    "stream_id"
                ].nunique()
            )

        else:

            unique_alerts = len(
                alert_history
            )

        history_column2.metric(
            "Unique Transactions",
            unique_alerts
        )

        if "risk_level" in alert_history.columns:

            st.subheader(
                "Notifications by Risk Level"
            )

            notification_counts = (
                alert_history[
                    "risk_level"
                ]
                .value_counts()
            )

            st.bar_chart(
                notification_counts
            )

        st.subheader(
            "Complete Alert History"
        )

        st.dataframe(
            alert_history,
            use_container_width=True,
            hide_index=True
        )


elif selected_page == "Model Performance":

    st.title(
        "Fraud Model Performance"
    )

    if model_metrics.empty:

        st.info(
            "Model metrics file was not found."
        )

    else:

        required_metric_columns = [
            "metric",
            "value"
        ]

        if all(
            column in model_metrics.columns
            for column in required_metric_columns
        ):

            metric_dictionary = dict(
                zip(
                    model_metrics["metric"],
                    model_metrics["value"]
                )
            )

            metric_columns = st.columns(
                len(metric_dictionary)
            )

            for metric_column, (
                metric_name,
                metric_value
            ) in zip(
                metric_columns,
                metric_dictionary.items()
            ):

                metric_column.metric(
                    metric_name,
                    f"{float(metric_value):.4f}"
                )

            st.subheader(
                "Metric Comparison"
            )

            chart_data = (
                model_metrics
                .set_index("metric")[
                    "value"
                ]
            )

            st.bar_chart(
                chart_data
            )

        else:

            st.dataframe(
                model_metrics,
                use_container_width=True,
                hide_index=True
            )

st.markdown("---")

st.caption(
    "Financial Fraud Detection and "
    "Risk Monitoring System"
)

Writing fraud_dashboard.py


In [4]:
from pathlib import Path

dashboard_file = Path(
    "fraud_dashboard.py"
)

print(
    "Dashboard file exists:",
    dashboard_file.exists()
)

print(
    "File size:",
    dashboard_file.stat().st_size
    if dashboard_file.exists()
    else 0
)

Dashboard file exists: True
File size: 12863


In [5]:
import subprocess
import sys

dashboard_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "fraud_dashboard.py"
    ]
)

print(
    "Dashboard started."
)

print(
    "Open http://localhost:8501"
)

Dashboard started.
Open http://localhost:8501


In [6]:
# TO STOP IT LATER:- 
dashboard_process.terminate()

print("Dashboard stopped.")

Dashboard stopped.
